## **Credit Card Fraud Detection**

### **Importing Essential Libraries**

In [59]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import scipy.stats as stats
import tensorflow as tf
import sklearn
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from tensorflow import keras
from tensorflow.keras import layers, models
from sklearn.impute import KNNImputer
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report,accuracy_score
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.svm import SVC
from sklearn.neighbors import KNeighborsClassifier
from xgboost import XGBClassifier
from lightgbm import LGBMClassifier
from catboost import CatBoostClassifier
import joblib
from imblearn.over_sampling import SMOTE
from sklearn.metrics import confusion_matrix


### **Dataset Loading**

In [60]:
df=pd.read_csv('creditcard.csv')

In [61]:
df.head(5)

,Time,V1,V2,V3,V4,V5,V6,V7,V8,V9,...,V21,V22,V23,V24,V25,V26,V27,V28,Amount,Class
0,0.0,-1.359807,-0.072781,2.536347,1.378155,-0.338321,0.462388,0.239599,0.098698,0.363787,...,-0.018307,0.277838,-0.110474,0.066928,0.128539,-0.189115,0.133558,-0.021053,149.62,0
1,0.0,1.191857,0.266151,0.166480,0.448154,0.060018,-0.082361,-0.078803,0.085102,-0.255425,...,-0.225775,-0.638672,0.101288,-0.339846,0.167170,0.125895,-0.008983,0.014724,2.69,0
2,1.0,-1.358354,-1.340163,1.773209,0.379780,-0.503198,1.800499,0.791461,0.247676,-1.514654,...,0.247998,0.771679,0.909412,-0.689281,-0.327642,-0.139097,-0.055353,-0.059752,378.66,0
3,1.0,-0.966272,-0.185226,1.792993,-0.863291,-0.010309,1.247203,0.237609,0.377436,-1.387024,...,-0.108300,0.005274,-0.190321,-1.175575,0.647376,-0.221929,0.062723,0.061458,123.50,0
4,2.0,-1.158233,0.877737,1.548718,0.403034,-0.407193,0.095921,0.592941,-0.270533,0.817739,...,-0.009431,0.798278,-0.137458,0.141267,-0.206010,0.502292,0.219422,0.215153,69.99,0


In [62]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 284807 entries, 0 to 284806
Data columns (total 31 columns):
 #   Column  Non-Null Count   Dtype  
---  ------  --------------   -----  
 0   Time    284807 non-null  float64
 1   V1      284807 non-null  float64
 2   V2      284807 non-null  float64
 3   V3      284807 non-null  float64
 4   V4      284807 non-null  float64
 5   V5      284807 non-null  float64
 6   V6      284807 non-null  float64
 7   V7      284807 non-null  float64
 8   V8      284807 non-null  float64
 9   V9      284807 non-null  float64
 10  V10     284807 non-null  float64
 11  V11     284807 non-null  float64
 12  V12     284807 non-null  float64
 13  V13     284807 non-null  float64
 14  V14     284807 non-null  float64
 15  V15     284807 non-null  float64
 16  V16     284807 non-null  float64
 17  V17     284807 non-null  float64
 18  V18     284807 non-null  float64
 19  V19     284807 non-null  float64
 20  V20     284807 non-null  float64
 21  V21     28

### **Data Preprocessing**

In [63]:
df.isnull().sum

<bound method DataFrame.sum of          Time     V1     V2     V3     V4     V5     V6     V7     V8     V9  \
0       False  False  False  False  False  False  False  False  False  False   
1       False  False  False  False  False  False  False  False  False  False   
2       False  False  False  False  False  False  False  False  False  False   
3       False  False  False  False  False  False  False  False  False  False   
4       False  False  False  False  False  False  False  False  False  False   
...       ...    ...    ...    ...    ...    ...    ...    ...    ...    ...   
284802  False  False  False  False  False  False  False  False  False  False   
284803  False  False  False  False  False  False  False  False  False  False   
284804  False  False  False  False  False  False  False  False  False  False   
284805  False  False  False  False  False  False  False  False  False  False   
284806  False  False  False  False  False  False  False  False  False  False   

        

In [64]:
scaler = StandardScaler()
df["Amount"] = scaler.fit_transform(df[["Amount"]])
df["Time"] = scaler.fit_transform(df[["Time"]])

### **Extracting Dataset into features and labels**

In [65]:
X = df.drop("Class", axis=1)
y = df["Class"]

### **Dividing into train-test-validation Dataset**

In [66]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)
X_train, X_val, y_train, y_val = train_test_split(X_train, y_train, test_size=0.25, random_state=42, stratify=y_train)

In [67]:
print(f"X_train : {X_train.shape}")
print(f"y_train : {y_train.shape}")
print(f"X_val : {X_val.shape}")
print(f"y_val : {y_val.shape}")
print(f"X_test : {X_test.shape}")
print(f"y_test : {y_test.shape}")

X_train : (170883, 30)
y_train : (170883,)
X_val : (56962, 30)
y_val : (56962,)
X_test : (56962, 30)
y_test : (56962,)


### **Baseline Model**

In [68]:
baseline_model=LogisticRegression(class_weight='balanced',max_iter=1000, random_state=42)

### **Baseline Model Training**

In [69]:
baseline_model.fit(X_train,y_train)

,penalty,'l2'
,dual,False
,tol,0.0001
,C,1.0
,fit_intercept,True
,intercept_scaling,1
,class_weight,'balanced'
,random_state,42
,solver,'lbfgs'
,max_iter,1000
,multi_class,'deprecated'


### **Baseline Model Prediction**

In [70]:
y_val_pred=baseline_model.predict(X_val)
y_test_pred = baseline_model.predict(X_test)

In [71]:
print("Accuracy on Validation Dataset")
print("Accuracy:", accuracy_score(y_val,y_val_pred))
print("\nClassification Report:\n", classification_report(y_val,y_val_pred))

Accuracy on Validation Dataset
Accuracy: 0.9749482110880938

Classification Report:
               precision    recall  f1-score   support

           0       1.00      0.98      0.99     56863
           1       0.06      0.90      0.11        99

    accuracy                           0.97     56962
   macro avg       0.53      0.94      0.55     56962
weighted avg       1.00      0.97      0.99     56962



In [72]:
print("Accuracy on Test Dataset")
print("Accuracy:", accuracy_score(y_test, y_test_pred))
print("\nClassification Report:\n", classification_report(y_test, y_test_pred))

Accuracy on Test Dataset
Accuracy: 0.975650433622415

Classification Report:
               precision    recall  f1-score   support

           0       1.00      0.98      0.99     56864
           1       0.06      0.92      0.11        98

    accuracy                           0.98     56962
   macro avg       0.53      0.95      0.55     56962
weighted avg       1.00      0.98      0.99     56962



### **Random Forest**

In [73]:
randomforest_model=RandomForestClassifier(class_weight='balanced',n_estimators=1000, random_state=42)

### **Random Forest Model Training**

In [74]:
randomforest_model.fit(X_train,y_train)

,n_estimators,1000
,criterion,'gini'
,max_depth,None
,min_samples_split,2
,min_samples_leaf,1
,min_weight_fraction_leaf,0.0
,max_features,'sqrt'
,max_leaf_nodes,None
,min_impurity_decrease,0.0
,bootstrap,True
,oob_score,False


### **Random Forest Model Prediction**

In [75]:
y_val_pred_r=randomforest_model.predict(X_val)
y_test_pred_r = randomforest_model.predict(X_test)

In [76]:
print("Accuracy on Validation Dataset")
print("Accuracy:", accuracy_score(y_val,y_val_pred_r))
print("\nClassification Report:\n", classification_report(y_val,y_val_pred_r))

Accuracy on Validation Dataset
Accuracy: 0.9994382219725431

Classification Report:
               precision    recall  f1-score   support

           0       1.00      1.00      1.00     56863
           1       0.96      0.71      0.81        99

    accuracy                           1.00     56962
   macro avg       0.98      0.85      0.91     56962
weighted avg       1.00      1.00      1.00     56962



In [77]:
print("Accuracy on Test Dataset")
print("Accuracy:", accuracy_score(y_test, y_test_pred_r))
print("\nClassification Report:\n", classification_report(y_test, y_test_pred_r))

Accuracy on Test Dataset
Accuracy: 0.9994733330992591

Classification Report:
               precision    recall  f1-score   support

           0       1.00      1.00      1.00     56864
           1       0.96      0.72      0.83        98

    accuracy                           1.00     56962
   macro avg       0.98      0.86      0.91     56962
weighted avg       1.00      1.00      1.00     56962



### **XG Boost Model**

In [78]:
neg = sum(y_train == 0)
pos = sum(y_train == 1)
scale_pos_weight = neg / pos
print(scale_pos_weight)

578.264406779661


In [79]:
xgb = XGBClassifier(n_estimators=300,max_depth=4,learning_rate=0.05,subsample=0.8,colsample_bytree=0.8,scale_pos_weight=1.875,random_state=42)


### **XG Boost Model Training**

In [80]:
xgb.fit(X_train, y_train)

,objective,'binary:logistic'
,base_score,None
,booster,None
,callbacks,None
,colsample_bylevel,None
,colsample_bynode,None
,colsample_bytree,0.8
,device,None
,early_stopping_rounds,None
,enable_categorical,False
,eval_metric,None


### **XGBoost Model Prediction**

In [81]:
y_val_pred_x=xgb.predict(X_val)
y_test_pred_x = xgb.predict(X_test)

In [82]:
print("Accuracy on Validation Dataset")
print("Accuracy:", accuracy_score(y_val,y_val_pred_x))
print("\nClassification Report:\n", classification_report(y_val,y_val_pred_x))

Accuracy on Validation Dataset
Accuracy: 0.9995084442259752

Classification Report:
               precision    recall  f1-score   support

           0       1.00      1.00      1.00     56863
           1       0.95      0.76      0.84        99

    accuracy                           1.00     56962
   macro avg       0.97      0.88      0.92     56962
weighted avg       1.00      1.00      1.00     56962



In [83]:
print("Accuracy on Test Dataset")
print("Accuracy:", accuracy_score(y_test, y_test_pred_x))
print("\nClassification Report:\n", classification_report(y_test, y_test_pred_x))

Accuracy on Test Dataset
Accuracy: 0.9995259997893332

Classification Report:
               precision    recall  f1-score   support

           0       1.00      1.00      1.00     56864
           1       0.91      0.81      0.85        98

    accuracy                           1.00     56962
   macro avg       0.95      0.90      0.93     56962
weighted avg       1.00      1.00      1.00     56962



### **Model Saving**

In [ ]:
joblib.dump(xgb, 'fraud_detection_model.pkl')
joblib.dump(scaler, "scaler.pkl")

['scaler.pkl']